In [16]:
import altair as alt
import pandas as pd
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

def d(year):
    return pd.Timestamp(f"{year}-01-01")

# ---- Data: Statistics Jersey projections, +400 net migration (Tables 10, 14, 16) ---
years = [2024, 2030, 2040, 2050, 2060, 2070, 2080]
df = pd.DataFrame({
    "date": [d(y) for y in years],
    "Under 16": [15410, 13590, 11690, 11350, 10650, 9900, 9510],
    "Aged 80+": [5770, 6920, 8540, 11310, 12230, 12820, 13210],
})
long = df.melt("date", var_name="series", value_name="value")

# Monthly interpolation so the shading flips colour exactly where the lines cross
monthly = df.set_index("date").resample("MS").interpolate("linear").reset_index()

# ---- Chart -----------------------------------------------------------------------
Y_TITLE = "People"
y_scale = alt.Scale(domain=[0, 18000])

x = alt.X("date:T", title=None, scale=alt.Scale(domain=[d(2024), d(2087)]),
          axis=alt.Axis(format="%Y", values=[alt.DateTime(year=y) for y in years]))
def y(field):
    return alt.Y(f"{field}:Q", title=Y_TITLE, scale=y_scale, axis=alt.Axis(format=",.0f"))

colour = alt.Color("series:N", legend=None)  # theme assigns the palette

def tag(chart, series):
    return chart.transform_calculate(series=f"'{series}'")

def shade(condition, series):
    return tag(alt.Chart(monthly).transform_filter(condition), series).mark_area(
        opacity=0.18
    ).encode(x=x, y=y("Aged 80+"), y2="Under 16:Q", color=colour)

more_children = shade("datum['Under 16'] >= datum['Aged 80+']", "Under 16")
more_over80 = shade("datum['Under 16'] < datum['Aged 80+']", "Aged 80+")

lines = alt.Chart(long).mark_line(strokeWidth=1.8).encode(
    x=x, y=y("value"), color=colour
)

# End-of-line direct labels instead of a legend
labels = alt.Chart(long).mark_text(align="left", dx=6, fontWeight="bold").encode(
    x=alt.X("max(date):T", title=None),
    y=alt.Y("value:Q", aggregate={"argmax": "date"}, title=Y_TITLE, scale=y_scale),
    text="series:N",
    color=colour,
)

# Starting values
def start_label(series, value, dy):
    row = pd.DataFrame({"date": [d(2024)], "value": [value], "text": [f"{value:,}"]})
    return tag(alt.Chart(row), series).mark_text(
        align="left", dx=2, dy=dy, fontWeight="bold", fontSize=14
    ).encode(x=x, y=y("value"), text="text:N", color=colour)

start_u16 = start_label("Under 16", 15410, -14)
start_80 = start_label("Aged 80+", 5770, 14)

# Where they meet
meet = alt.Chart(pd.DataFrame({
})).mark_text(fontWeight="bold", opacity=0.85).encode(
    x=x, y=y("value"), text="text:N"
)

chart = (more_children + more_over80 + lines + labels + meet).properties(
    width=700, height=270,
    autosize=alt.AutoSizeParams(type="fit", contains="padding"),
)

chart.save("jersey_over80_u16.svg")
chart.save("jersey_over80_u16.png", ppi=288)
chart

alt.LayerChart(...)